## Gold Layer
- Read from the silver tables 
- left join crm tables iwth the erp tables converting 6 tables to 3 
- do sanity checks to ensure all joins are correct
- rename the tables and model into facts and dimensions

### Read tables from silver layer


In [0]:

df_crm_cust = spark.table("workspace.silver.crm_customer_info_cleaned")
df_erp_cust = spark.table("workspace.silver.erp_customer_info_cleaned")
df_erp_loc = spark.table("workspace.silver.erp_customer_location_cleaned")



In [0]:
from pyspark.sql.functions import coalesce, col

# 1. Join but rename the ERP gender immediately to avoid name collision
df_joined = df_crm_cust.join(
    df_erp_cust.withColumnRenamed("gender", "erp_gender"), 
    on='customer_key', 
    how='left'
)

# 2. Coalesce them into a single 'gender' column
df_joined = df_joined.withColumn("gender", coalesce(col("gender"), col("erp_gender")))

# 3. Now you can safely drop the temporary 'erp_gender'
df_joined = df_joined.drop("erp_gender")

df_joined.display()

In [0]:
# left join df_joined with df_erp_loc on customer_key
df_final = df_joined.join(df_erp_loc, on='customer_key', how='left')
df_final.display()

In [0]:
## write into golder layer 
df_final.write.mode("overwrite").saveAsTable("workspace.gold.dim_customer_info")